# Lab1 — ResNet18 vs ResNet50 on Google Colab
Run cells in order. First choose **Runtime → Change runtime type → GPU**, then connect.
Upload the prepared `Lab1_colab_code.zip` from your Mac in step 1. This works before your code is pushed to GitHub.
The notebook downloads Kaggle data directly into Colab, uses the same training split for both models, and saves checkpoints/plots/metrics to Google Drive.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

PROJECT = Path('/content/Lab1')
PROJECT.mkdir(exist_ok=True)
uploaded = files.upload()
if 'Lab1_colab_code.zip' not in uploaded:
    raise ValueError('Upload Lab1_colab_code.zip from your Lab1 folder.')
with zipfile.ZipFile('Lab1_colab_code.zip') as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError('Unexpected archive path')
    archive.extractall(PROJECT)
%cd /content/Lab1


## 2. Check GPU and install download utilities
Keep Colab's preinstalled matching PyTorch/torchvision packages. Stop here if no GPU is available.


In [ ]:
import sys
import subprocess
import torch
import torchvision

print('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU in Runtime → Change runtime type, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'matplotlib', 'Pillow', 'tqdm'], check=True)


## 3. Mount Google Drive and create a run folder
Allow Google Drive access when Colab prompts. Keep `RUN_DIR` unchanged while running the remaining cells. Each fresh execution of this cell creates a new folder; completed results from earlier runs remain saved. An interrupted training run must be restarted in a new folder; this version does not resume optimizer state.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone

drive.mount('/content/drive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = Path('/content/drive/MyDrive/Lab1/runs') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(freeze)
print('Results will be saved to:', RUN_DIR)


## 4. Download the Kaggle dataset
No manual dataset upload is needed. If Kaggle reports that authentication or consent is required, sign in to Kaggle, visit the dataset page, and store your API token in Colab Secrets as `KAGGLE_API_TOKEN` with notebook access enabled. Then rerun this cell. KaggleHub reads that secret; do not put tokens in notebook cells.
Dataset: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia


In [ ]:
import json
import kagglehub
from download_dataset import DATASET_HANDLE, find_dataset_root
from dataset import scan_split

downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
DATASET = find_dataset_root(downloaded)
metadata = {'handle': DATASET_HANDLE, 'download_directory': str(downloaded), 'dataset_root': str(DATASET)}
(RUN_DIR / 'dataset_location.json').write_text(json.dumps(metadata, indent=2))
print('Dataset root:', DATASET)
for split in ('train', 'val', 'test'):
    samples = scan_split(DATASET / split)
    print(split, {name: sum(label == index for _, label in samples) for index, name in enumerate(('NORMAL', 'PNEUMONIA'))})


## 5. Train ResNet18
Defaults: 20 epochs, batch size 32, seed 42, ImageNet pretrained weights. A stratified 20% of the supplied training images is used for validation; the supplied `val` folder is unused. The selected checkpoint maximizes validation F1. Final test evaluation happens after training, not every epoch.
If GPU memory is insufficient, use batch size 16 for both models in a fresh run folder.


In [ ]:
EPOCHS = 20
BATCH_SIZE = 32

def run_streamed(command, log_path):
    with Path(log_path).open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        if status:
            raise subprocess.CalledProcessError(status, command)

def train_model(name):
    command = [sys.executable, '-u', 'train.py', '--dataset', str(DATASET), '--model', name,
               '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', '42',
               '--workers', '2', '--device', 'cuda', '--output', str(RUN_DIR)]
    run_streamed(command, RUN_DIR / f'{name}_training.log')

train_model('resnet18')


## 6. Train ResNet50
Use the same configuration and dataset as ResNet18.


In [ ]:
train_model('resnet50')


## 7. Compare and display results
Screenshot the measured final test metrics for your report.


In [ ]:
from IPython.display import display, Image

subprocess.run([sys.executable, 'compare.py', '--runs', str(RUN_DIR / 'resnet18'), str(RUN_DIR / 'resnet50'),
                '--output', str(RUN_DIR / 'comparison')], check=True)
for name in ('resnet18', 'resnet50'):
    metrics = json.loads((RUN_DIR / name / 'test_metrics.json').read_text())
    print(name, json.dumps(metrics, indent=2))
    display(Image(filename=str(RUN_DIR / name / 'confusion_matrix.png')))
for filename in ('train_accuracy.png', 'train_f1.png', 'val_accuracy.png', 'val_f1.png'):
    display(Image(filename=str(RUN_DIR / 'comparison' / filename)))


## 8. Download results to your Mac
The ZIP includes both trained checkpoints, logs, curves, split manifests, and metrics. It remains on Google Drive as well. Extract into a new `results/colab_RUN_ID` folder in your local Lab1 directory. Commit plots/metrics/code to GitHub; `.pt` weights stay ignored. Use the actual results to complete `report_template.md`.


In [ ]:
import shutil
archive = shutil.make_archive(f'/content/Lab1_results_{RUN_ID}', 'zip', root_dir=RUN_DIR)
files.download(archive)
